In [1]:
import os
import cv2
import numpy as np
import pandas as pd
import tensorflow as tf
import tensorflow.keras.backend as K
from tqdm import tqdm
from scipy.ndimage import gaussian_filter
from sklearn.model_selection import train_test_split
from tensorflow.keras import layers, models, optimizers
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau

# --- الإعدادات العامة للمشروع ---
IMG_SIZE = (256, 256)
BATCH_SIZE = 8  # مقاس مثالي عشان الـ GPU ميفصلش
EPOCHS = 60
CSV_PATH = '/Users/emmyel-sawy/Desktop/Image Forgery Localization/final_manifest.csv' # عدل المسار لملفك الأساسي

2026-05-09 01:47:14.369783: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1778291234.564049     110 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1778291234.620929     110 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1778291235.096653     110 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1778291235.096698     110 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1778291235.096700     110 computation_placer.cc:177] computation placer alr

In [2]:
@tf.keras.utils.register_keras_serializable()
def dice_coef(y_true, y_pred, smooth=1e-5):
    y_true_f = tf.cast(K.flatten(y_true), tf.float32)
    y_pred_f = tf.cast(K.flatten(y_pred), tf.float32)
    intersection = K.sum(y_true_f * y_pred_f)
    return (2. * intersection + smooth) / (K.sum(y_true_f) + K.sum(y_pred_f) + smooth)

@tf.keras.utils.register_keras_serializable()
def focal_dice_loss(y_true, y_pred):
    y_true = tf.cast(y_true, tf.float32)
    y_pred = tf.cast(y_pred, tf.float32)
    
    # Dice Loss
    dice_loss = 1.0 - dice_coef(y_true, y_pred)
    
    # Focal Loss (للتركيز على التزوير الصغير)
    alpha = 0.25
    gamma = 2.0
    bce = K.binary_crossentropy(y_true, y_pred)
    bce_exp = K.exp(-bce)
    focal_loss = alpha * K.pow((1.0 - bce_exp), gamma) * bce
    
    return focal_loss + (dice_loss * 2.0) # وزن مضاعف للـ Dice

In [3]:
def extract_forensic_features(img_path):
    img = cv2.imread(img_path)
    if img is None: 
        return np.zeros((256, 256, 6), dtype=np.float32)
    
    img_res = cv2.resize(img, IMG_SIZE)
    rgb = cv2.cvtColor(img_res, cv2.COLOR_BGR2RGB) / 255.0
    
    # 1. ELA Map
    temp = 'tmp.jpg'
    cv2.imwrite(temp, img_res, [cv2.IMWRITE_JPEG_QUALITY, 90])
    ela = cv2.absdiff(img_res, cv2.imread(temp))
    ela = cv2.cvtColor(ela, cv2.COLOR_BGR2GRAY).astype(np.float32) / 255.0
    if os.path.exists(temp): os.remove(temp)
        
    # 2. Noise Residual
    gray = cv2.cvtColor(img_res, cv2.COLOR_BGR2GRAY).astype(np.float32) / 255.0
    noise = (gray - gaussian_filter(gray, sigma=2))
    noise = (noise - np.min(noise)) / (np.max(noise) - np.min(noise) + 1e-7)
    
    # 3. Edge Map
    edge = cv2.Canny(img_res, 100, 200).astype(np.float32) / 255.0
    
    return np.concatenate([rgb, np.expand_dims(ela, -1), 
                           np.expand_dims(noise, -1), 
                           np.expand_dims(edge, -1)], axis=-1)

In [4]:
class ForgeryDataGenerator(tf.keras.utils.Sequence):
    def __init__(self, df, batch_size=8, shuffle=True):
        self.df = df.reset_index(drop=True)
        self.batch_size = batch_size
        self.shuffle = shuffle
        self.indices = np.arange(len(self.df))
        if self.shuffle:
            np.random.shuffle(self.indices)
            
    def __len__(self):
        return int(np.ceil(len(self.df) / self.batch_size))
    
    def __getitem__(self, idx):
        batch_indices = self.indices[idx * self.batch_size : (idx + 1) * self.batch_size]
        batch_df = self.df.iloc[batch_indices]
        
        X, Y = [], []
        for _, row in batch_df.iterrows():
            feat = extract_forensic_features(row['image_path'])
            mask = cv2.imread(row['mask_path'], cv2.IMREAD_GRAYSCALE)
            if mask is None: mask = np.zeros(IMG_SIZE)
            mask = cv2.resize(mask, IMG_SIZE)
            
            # تجهيز الماسك
            mask_expanded = np.expand_dims((mask > 127).astype(np.float32), -1)
            
            # 🔥 السلاح الأول: Data Augmentation (بيحصل في التدريب بس عشان self.shuffle بتكون True)
            if self.shuffle:
                if np.random.rand() > 0.5: # تقليب أفقي بنسبة 50%
                    feat = np.fliplr(feat)
                    mask_expanded = np.fliplr(mask_expanded)
                if np.random.rand() > 0.5: # تقليب رأسي بنسبة 50%
                    feat = np.flipud(feat)
                    mask_expanded = np.flipud(mask_expanded)
            
            X.append(feat)
            Y.append(mask_expanded)
            
        return np.array(X), np.array(Y)
    
    def on_epoch_end(self):
        if self.shuffle:
            np.random.shuffle(self.indices)

In [5]:
def build_unet_efficientnet():
    # 1. المدخلات (6 قنوات)
    inputs = layers.Input(shape=(256, 256, 6), name='custom_6ch_input')
    
    # 2. Bottleneck
    x = layers.Conv2D(3, (3, 3), padding='same', name='channel_bottleneck')(inputs)
    s1 = x 
    
    # 3. بناء الـ EfficientNet
    base_model = tf.keras.applications.EfficientNetB0(
        include_top=False, 
        weights='imagenet', 
        input_shape=(256, 256, 3)
    )
    
    # 4. الـ Extractor
    skip_names = [
        "block2a_expand_activation",
        "block3a_expand_activation",
        "block4a_expand_activation",
        "block6a_expand_activation" 
    ]
    extractor_outputs = [base_model.get_layer(name).output for name in skip_names]
    extractor = models.Model(inputs=base_model.input, outputs=extractor_outputs, name='efficientnet_extractor')
    
    s2, s3, s4, b1 = extractor(x)

    # 5. Decoder Block (مع تفعيل الـ Dropout 🔥)
    def decoder_block(input_tensor, skip_tensor, filters, dropout_rate=0.0):
        x = layers.UpSampling2D((2, 2))(input_tensor)
        x = layers.Conv2D(filters, 3, padding="same")(x)
        if x.shape[1] != skip_tensor.shape[1]: 
            skip_tensor = tf.image.resize(skip_tensor, [x.shape[1], x.shape[2]])
        x = layers.Concatenate()([x, skip_tensor])
        x = layers.Conv2D(filters, 3, padding="same", activation="relu")(x)
        x = layers.BatchNormalization()(x)
        
        # إضافة طبقة الـ Dropout هنا لمنع الحفظ
        if dropout_rate > 0:
            x = layers.Dropout(dropout_rate)(x)
            
        return x

    # 6. طريق العودة (بنسب Dropout متدرجة)
    d1 = decoder_block(b1, s4, 256, dropout_rate=0.4)
    d2 = decoder_block(d1, s3, 128, dropout_rate=0.3)
    d3 = decoder_block(d2, s2, 64,  dropout_rate=0.2)
    d4 = decoder_block(d3, s1, 32,  dropout_rate=0.1)
    
    outputs = layers.Conv2D(1, (1, 1), padding="same", activation="sigmoid")(d4)
    return models.Model(inputs, outputs)

In [6]:
import matplotlib.pyplot as plt
from IPython.display import clear_output
import tensorflow as tf

class PaperReadyPlotCallback(tf.keras.callbacks.Callback):
    def on_train_begin(self, logs=None):
        # تجهيز قواميس لحفظ الأرقام
        self.history_dict = {'loss': [], 'val_loss': [], 'dice_coef': [], 'val_dice_coef': []}

    def on_epoch_end(self, epoch, logs=None):
        # مسح الرسمة القديمة عشان نرسم الجديدة لايف
        clear_output(wait=True)
        
        # تخزين أرقام الإيبوك الحالي (استخدمنا get عشان نتفادى أي أخطاء لو المقياس مش موجود)
        self.history_dict['loss'].append(logs.get('loss'))
        self.history_dict['val_loss'].append(logs.get('val_loss'))
        self.history_dict['dice_coef'].append(logs.get('dice_coef'))
        self.history_dict['val_dice_coef'].append(logs.get('val_dice_coef'))
        
        epochs_range = range(1, len(self.history_dict['loss']) + 1)
        
        # إنشاء اللوحة للبيبر
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))
        plt.style.use('default') 
        
        # ----------------- رسمة الأداء (Dice) -----------------
        ax1.plot(epochs_range, self.history_dict['dice_coef'], label='Train Dice', color='#1f77b4', linewidth=2.5)
        ax1.plot(epochs_range, self.history_dict['val_dice_coef'], label='Val Dice', color='#ff7f0e', linewidth=2.5)
        ax1.set_title('Model Performance (Dice Score)', fontsize=14, weight='bold')
        ax1.set_xlabel('Epochs', fontsize=12, weight='bold')
        ax1.set_ylabel('Score', fontsize=12, weight='bold')
        ax1.legend(loc='lower right', fontsize=11)
        ax1.grid(True, linestyle='--', alpha=0.6)
        
        # ----------------- رسمة الخسارة (Loss) -----------------
        ax2.plot(epochs_range, self.history_dict['loss'], label='Train Loss', color='#1f77b4', linewidth=2.5)
        ax2.plot(epochs_range, self.history_dict['val_loss'], label='Val Loss', color='#ff7f0e', linewidth=2.5)
        ax2.set_title('Model Loss (Convergence)', fontsize=14, weight='bold')
        ax2.set_xlabel('Epochs', fontsize=12, weight='bold')
        ax2.set_ylabel('Loss', fontsize=12, weight='bold')
        ax2.legend(loc='upper right', fontsize=11)
        ax2.grid(True, linestyle='--', alpha=0.6)
        
        # الحفظ والعرض
        plt.tight_layout()
        plt.savefig('paper_training_curves.png', dpi=300, bbox_inches='tight')
        plt.show()

In [7]:
# 1. قراءة وتقسيم البيانات
df = pd.read_csv(CSV_PATH, encoding='latin-1')
train_df, temp_df = train_test_split(df, test_size=0.30, random_state=42)
val_df, test_df = train_test_split(temp_df, test_size=0.50, random_state=42)

print(f"✅ Data: Train={len(train_df)}, Val={len(val_df)}, Test={len(test_df)}")

# 2. تهيئة الـ Generators
train_gen = ForgeryDataGenerator(train_df, batch_size=BATCH_SIZE, shuffle=True)
val_gen = ForgeryDataGenerator(val_df, batch_size=BATCH_SIZE, shuffle=False)

# 3. بناء وتجميع الموديل
model = build_unet_efficientnet()

# 🔥 السلاح الثالث: استخدام AdamW بدل Adam لمعاقبة الأوزان الكبيرة (Weight Decay)
model.compile(
    optimizer=tf.keras.optimizers.AdamW(learning_rate=1e-4, weight_decay=1e-4), 
    loss=focal_dice_loss, 
    metrics=['accuracy', dice_coef]
)

# 4. الـ Callbacks للحفاظ على أفضل نسخة وإضافة الرسومات اللايف
callbacks = [
    ModelCheckpoint('ultimate_forgery_model_v2.keras', monitor='val_dice_coef', save_best_only=True, mode='max'),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=4, min_lr=1e-6),
    EarlyStopping(monitor='val_loss', patience=12, restore_best_weights=True),
    PaperReadyPlotCallback()  # 👈 إضافة الكول باك بتاع الرسومات للبيبر
]

# 5. الانطلاق 
print("Starting Anti-Overfitting Training...")
history = model.fit(train_gen, validation_data=val_gen, epochs=EPOCHS, callbacks=callbacks)